In [0]:
%sql
-- ============================================================
-- VALIDATION 1: Record counts across all layers
-- Confirms no data loss Bronze → Silver → Gold
-- ============================================================

SELECT 'Bronze - Transactions' AS layer_table, COUNT(*) AS record_count FROM bronze_it_finance.raw_vendor_transactions
UNION ALL
SELECT 'Bronze - Cost Centers', COUNT(*) FROM bronze_it_finance.raw_cost_centers
UNION ALL
SELECT 'Silver - Transactions', COUNT(*) FROM silver_it_finance.clean_vendor_transactions
UNION ALL
SELECT 'Silver - Cost Centers', COUNT(*) FROM silver_it_finance.clean_cost_centers
UNION ALL
SELECT 'Gold - Fact Spend', COUNT(*) FROM gold_it_finance.fact_it_spend_monthly
UNION ALL
SELECT 'Gold - Dim Vendor', COUNT(*) FROM gold_it_finance.dim_vendor
UNION ALL
SELECT 'Gold - Dim Cost Center', COUNT(*) FROM gold_it_finance.dim_cost_center
ORDER BY layer_table

layer_table,record_count
Bronze - Cost Centers,5
Bronze - Transactions,10
Gold - Dim Cost Center,5
Gold - Dim Vendor,6
Gold - Fact Spend,10
Silver - Cost Centers,5
Silver - Transactions,10


In [0]:
%sql
-- ============================================================
-- VALIDATION 2: Star Schema Join - Spend by Cost Center & Vendor
-- Confirms fact table joins cleanly to both dimensions
-- ============================================================

SELECT 
    cc.cost_center_name,
    cc.department,
    v.vendor,
    v.spend_category,
    f.fiscal_year,
    f.fiscal_month,
    f.forecast_cycle,
    f.total_spend
FROM gold_it_finance.fact_it_spend_monthly f
JOIN gold_it_finance.dim_cost_center cc ON f.cost_center_key = cc.cost_center_key
JOIN gold_it_finance.dim_vendor v ON f.vendor_key = v.vendor_key
ORDER BY f.fiscal_year, f.fiscal_month, f.total_spend DESC

cost_center_name,department,vendor,spend_category,fiscal_year,fiscal_month,forecast_cycle,total_spend
Cloud Operations,Infrastructure,AWS,Cloud Infrastructure,2026,1,3+9,142500.00
Cloud Operations,Infrastructure,Microsoft Azure,Cloud Infrastructure,2026,1,3+9,98200.00
Enterprise Applications,Corporate IT,ServiceNow,ITSM Platform,2026,1,3+9,76800.00
Cybersecurity,Infrastructure,CrowdStrike,Security Software,2026,1,3+9,54000.00
Data & Analytics,Corporate IT,Databricks,Data Platform,2026,1,3+9,38400.00
Cloud Operations,Infrastructure,AWS,Cloud Infrastructure,2026,2,3+9,158900.00
Cybersecurity,Infrastructure,CrowdStrike,Security Software,2026,2,3+9,54000.00
Data & Analytics,Corporate IT,Databricks,Data Platform,2026,2,3+9,41200.00
Enterprise Applications,Corporate IT,Salesforce,CRM Platform,2026,2,3+9,29500.00
Cloud Operations,Infrastructure,Microsoft Azure,Cloud Infrastructure,2026,3,3+9,103400.00


In [0]:
%sql
-- ============================================================
-- VALIDATION 3: Executive Spend Summary
-- Total IT spend by department and forecast cycle
-- ============================================================

SELECT
    cc.department,
    f.forecast_cycle,
    f.fiscal_year,
    SUM(f.total_spend) AS total_spend,
    COUNT(DISTINCT v.vendor) AS vendor_count
FROM gold_it_finance.fact_it_spend_monthly f
JOIN gold_it_finance.dim_cost_center cc ON f.cost_center_key = cc.cost_center_key
JOIN gold_it_finance.dim_vendor v ON f.vendor_key = v.vendor_key
GROUP BY cc.department, f.forecast_cycle, f.fiscal_year
ORDER BY total_spend DESC

department,forecast_cycle,fiscal_year,total_spend,vendor_count
Infrastructure,3+9,2026,611000.00,3
Corporate IT,3+9,2026,185900.00,3
